In [ ]:
###############################################################################
# IMPORTS
###############################################################################

import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.backends.backend_pdf import PdfPages

warnings.filterwarnings('ignore')

In [ ]:
###############################################################################
# MATPLOTLIB STYLE AND COLOR CONSTANTS
###############################################################################

# Journal-quality matplotlib settings
plt.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset':  'stix',
    'font.size':         10,
    'axes.labelsize':    12,
    'axes.titlesize':    13,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'legend.fontsize':   10,
    'figure.dpi':        300,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'axes.linewidth':    0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction':   'in',
    'ytick.direction':   'in',
    'xtick.top':         True,
    'ytick.right':       True,
})

OH_COLOR  = '#D62728'
BIN_COLOR = '#1F77B4'
OH_MARKER  = 's'
BIN_MARKER = 's' 

In [ ]:
###############################################################################
# TIMING CONSTANTS AND DATA LOADING
###############################################################################

# Timing constants (must match the benchmark run that produced the CSV).
T_ANNEAL  = 20.0                  # μs per anneal
T_READOUT = 100.0                 # μs per readout
T_READ    = T_ANNEAL + T_READOUT  # 120 μs per read
R         = 1000                  # reads per problem

# Right-censoring threshold: total wall-clock budget per instance.
# Instances that never found the best energy are censored at this value.
T_CENSOR  = R * T_READ

# ── Input / output paths ──────────────────────────────────────────────────────
BENCHMARK_CSV = ('<INPUT BENCHMARK RESULTS CSV HERE>')
EMBEDDING_CSV = ('<INPUT EMBEDDING RESULTS CSV HERE>')
PDF_OUT       = ('<INPUT PDF OUTPUT PATH HERE>')

df           = pd.read_csv(BENCHMARK_CSV)
df_embedding = pd.read_csv(EMBEDDING_CSV)

print(f'Loaded {len(df)} benchmark instances')
print(f'T_read  = {T_READ} μs   R = {R}   T_censor = {T_CENSOR} μs')
print(f'oh_tts_us:  {df["oh_tts_us"].notna().sum()} observed, '
      f'{df["oh_tts_us"].isna().sum()} censored')
print(f'bin_tts_us: {df["bin_tts_us"].notna().sum()} observed, '
      f'{df["bin_tts_us"].isna().sum()} censored')

In [ ]:
###############################################################################
# KAPLAN-MEIER MEDIAN TTS WITH GREENWOOD 95 % CI
###############################################################################

def kaplan_meier_median_ci(times, events):

    """
    Compute Kaplan-Meier median TTS and its 95 % confidence interval.

    Each instance contributes either an observed TTS (event=1) or a right-
    censored observation at T_CENSOR (event=0).  The survival function
    S_hat(t) is estimated via the Kaplan-Meier product-limit estimator;
    the median is read off at S_hat(t) = 0.5 and pointwise 95 % confidence
    limits on S are obtained via Greenwood's formula, then inverted to
    yield a CI on the median time.

    Parameters
    ----------
    times : array-like
        Per-instance times: observed TTS for uncensored instances,
        T_CENSOR for right-censored instances.
    events : array-like
        1 if the instance found the target solution (event), 0 if the
        instance is right-censored.

    Returns
    -------
    median : float
        KM median TTS.  np.inf if S_hat(t) never reaches 0.5.
    ci_lo : float
        Lower 95 % CI on the median time.
    ci_hi : float
        Upper 95 % CI on the median time.
    is_lower_bound : bool
        True if the survival curve never crossed 0.5 and the reported
        median is therefore a lower bound only.
    """

    times  = np.asarray(times,  dtype=float)
    events = np.asarray(events, dtype=int)
    n_total = len(times)
    if n_total == 0:
        return np.nan, np.nan, np.nan, True

    # Sort by time; at ties, events precede censored (standard KM convention)
    order    = np.lexsort((-events, times))
    t_sorted = times[order]
    e_sorted = events[order]

    event_times = np.unique(t_sorted[e_sorted == 1])
    if len(event_times) == 0:
        return np.inf, np.inf, np.inf, True

    S             = 1.0
    greenwood_sum = 0.0
    n_at_risk     = n_total

    km_times, km_S, km_var = [], [], []

    idx = 0
    for t_j in event_times:
        # Drop subjects censored strictly before t_j
        while idx < n_total and (t_sorted[idx] < t_j or
               (t_sorted[idx] == t_j and e_sorted[idx] == 0)):
            if t_sorted[idx] < t_j:
                n_at_risk -= 1
            elif t_sorted[idx] == t_j and e_sorted[idx] == 0:
                break
            idx += 1

        # Count events and censored at t_j
        d_j = 0
        c_j = 0
        while idx < n_total and t_sorted[idx] == t_j:
            if e_sorted[idx] == 1:
                d_j += 1
            else:
                c_j += 1
            idx += 1

        if n_at_risk > 0 and d_j > 0:
            S *= (1.0 - d_j / n_at_risk)
            greenwood_sum += (d_j / (n_at_risk * (n_at_risk - d_j))
                              if n_at_risk > d_j else 0.0)

        km_times.append(t_j)
        km_S.append(S)
        km_var.append(S**2 * greenwood_sum)

        n_at_risk -= (d_j + c_j)

    km_times = np.array(km_times)
    km_S     = np.array(km_S)
    km_var   = np.array(km_var)

    # Median: first event time at which S <= 0.5
    cross = np.where(km_S <= 0.5)[0]
    if len(cross) == 0:
        return km_times[-1], km_times[-1], np.inf, True
    median = km_times[cross[0]]

    # Pointwise 95 % CI on S, inverted to CI on median
    se_S = np.sqrt(km_var)
    S_lo = km_S - 1.96 * se_S   # lower CI on S -> upper CI on median
    S_hi = km_S + 1.96 * se_S   # upper CI on S -> lower CI on median

    cross_lo = np.where(S_hi <= 0.5)[0]
    ci_lo = km_times[cross_lo[0]] if len(cross_lo) > 0 else km_times[0]

    cross_hi = np.where(S_lo <= 0.5)[0]
    ci_hi = km_times[cross_hi[0]] if len(cross_hi) > 0 else np.inf

    return median, ci_lo, ci_hi, False


# ── Quick sanity check ────────────────────────────────────────────────────────
_times  = np.array([100, 200, 300, T_CENSOR, T_CENSOR])
_events = np.array([1, 1, 1, 0, 0])
_med, _lo, _hi, _lb = kaplan_meier_median_ci(_times, _events)
print(f'KM sanity check: median={_med}, CI=[{_lo}, {_hi}], lower_bound={_lb}')

In [ ]:
###############################################################################
# SURVIVAL-DATA PREPARATION AND PER-GROUP AGGREGATION
###############################################################################

def prepare_survival_data(df: pd.DataFrame, tts_col: str):

    """
    Build (time, event) arrays from a TTS column.

    Observed TTS -> event = 1, time = TTS.
    NaN TTS      -> event = 0 (right-censored), time = T_CENSOR.
    """

    observed = df[tts_col].notna()
    time  = np.where(observed, df[tts_col].values, T_CENSOR)
    event = observed.astype(int).values
    return time, event


def km_tts_by_group(df: pd.DataFrame, group_col: str, tts_col: str) -> pd.DataFrame:

    """
    Compute Kaplan-Meier median TTS + 95 % CI for each value of group_col.

    Returns a DataFrame with columns:
        group        -- unique value of group_col
        median       -- KM median TTS
        ci_lo, ci_hi -- 95 % CI on the median
        lower_bound  -- True if S never crossed 0.5
        n_total      -- instances in this group
        n_observed   -- instances that contributed an event (solved)
    """

    time_all, event_all = prepare_survival_data(df, tts_col)
    groups = sorted(df[group_col].unique())

    rows = []
    for g in groups:
        mask = df[group_col].values == g
        med, lo, hi, lb = kaplan_meier_median_ci(time_all[mask], event_all[mask])
        rows.append(dict(
            group=g, median=med, ci_lo=lo, ci_hi=hi,
            lower_bound=lb,
            n_total=int(mask.sum()),
            n_observed=int(event_all[mask].sum()),
        ))
    return pd.DataFrame(rows)

In [ ]:
###############################################################################
# GROUPED-MEAN HELPERS (FOR NON-TTS QUANTITIES)
###############################################################################

def grouped_mean_se(df: pd.DataFrame, group_col: str, val_col: str):

    """
    Return (xs, means, standard_errors) for val_col grouped by group_col.
    NaN values are dropped before aggregation.
    """

    groups = sorted(df[group_col].unique())
    xs, ys, ses = [], [], []
    for g in groups:
        vals = df.loc[df[group_col] == g, val_col].dropna()
        if len(vals) > 0:
            xs.append(g)
            ys.append(vals.mean())
            ses.append(vals.std(ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0.0)
    return np.array(xs), np.array(ys), np.array(ses)

In [ ]:
###############################################################################
# PLOTTING HELPERS
###############################################################################

def make_square_ax(fig=None):

    """Create a square-aspect matplotlib axes."""

    if fig is None:
        fig = plt.figure(figsize=(4.5, 4.5))
    ax = fig.add_subplot(111)
    ax.set_box_aspect(1)
    return fig, ax


def plot_km_tts_pair(ax, km_oh: pd.DataFrame, km_bin: pd.DataFrame,
                     xlabel: str, ylabel: str = r'Median TTS ($\mu$s)'):

    """
    Plot KM median TTS (with 95 % CI) for one-hot and logarithmic encodings.

    Finite medians are drawn as markers with asymmetric error bars.
    Groups whose survival curve never reached 0.5 (`lower_bound=True`) are
    plotted as upward arrows, indicating that the true median lies above.

    Parameters
    ----------
    ax : matplotlib Axes
        Target axes.
    km_oh, km_bin : DataFrame
        Outputs of km_tts_by_group for the one-hot and logarithmic encodings.
    xlabel : str
        Label for the x-axis (group variable).
    ylabel : str, optional
        Label for the y-axis. Defaults to 'Median TTS (μs)'.
    """

    plot_series = [
        (km_oh,  OH_COLOR,  OH_MARKER,  'One-Hot'),
        (km_bin, BIN_COLOR, BIN_MARKER, 'Logarithmic'),
    ]

    for km_df, color, marker, label in plot_series:
        finite = km_df[~km_df['lower_bound']].copy()
        lb     = km_df[ km_df['lower_bound']].copy()

        # ── Finite medians ────────────────────────────────────────────────────
        if len(finite) > 0:
            x = finite['group'].values
            y = finite['median'].values
            yerr_lo = np.clip(y - finite['ci_lo'].values, 0, None)
            yerr_hi_raw = finite['ci_hi'].values
            # Clip infinite upper CI to a visible cap (censoring horizon)
            yerr_hi = np.where(np.isinf(yerr_hi_raw),
                               T_CENSOR - y,
                               yerr_hi_raw - y)
            yerr_hi = np.clip(yerr_hi, 0, None)

            ax.errorbar(x, y,
                        yerr=[yerr_lo, yerr_hi],
                        fmt=marker, color=color,
                        markeredgecolor='black', markeredgewidth=0.8,
                        markersize=6, ecolor='black', elinewidth=0.9,
                        capsize=3, capthick=0.9, label=label, zorder=3)

        # ── Lower-bound points (survival never crossed 0.5) ───────────────────
        if len(lb) > 0:
            x_lb = lb['group'].values
            y_lb = lb['median'].values
            y_lb = np.where(np.isinf(y_lb), T_CENSOR, y_lb)
            lbl  = label if len(finite) == 0 else None
            ax.plot(x_lb, y_lb, marker=r'$\uparrow$', color=color,
                    linestyle='none', markersize=10,
                    markeredgecolor='black', markeredgewidth=0.3,
                    label=lbl, zorder=3)

    ax.set_yscale('log')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend(frameon=True, fancybox=False, edgecolor='black', framealpha=1.0)
    ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.yaxis.get_major_formatter().set_scientific(False)
    ax.yaxis.get_major_formatter().set_useOffset(False)


def plot_qubit_pair(ax, xvals_oh, yvals_oh, se_oh,
                    xvals_bin, yvals_bin, se_bin,
                    xlabel: str, ylabel: str = 'Logical Qubits',
                    title: str = None):

    """
    Plot a qubit-count quantity (or similar scalar metric) grouped on one axis
    for the one-hot and logarithmic encodings, with symmetric error bars.
    """

    ax.errorbar(xvals_oh, yvals_oh, yerr=se_oh,
                fmt=f'-{OH_MARKER}', color=OH_COLOR, label='One-Hot',
                markeredgecolor='black', markeredgewidth=0.8, markersize=6,
                ecolor='black', elinewidth=0.9, capsize=3, capthick=0.9,
                zorder=3)
    ax.errorbar(xvals_bin, yvals_bin, yerr=se_bin,
                fmt=f'-{BIN_MARKER}', color=BIN_COLOR,
                label='Logarithmic (Post-Quadratization)',
                markeredgecolor='black', markeredgewidth=0.8, markersize=6,
                ecolor='black', elinewidth=0.9, capsize=3, capthick=0.9,
                zorder=3)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    ax.legend()

In [ ]:
###############################################################################
# OPEN MULTI-PAGE PDF AND DEFINE GROUPING VARIABLES
###############################################################################

pdf = PdfPages(PDF_OUT)

# Grouping variables for benchmark-table plots
group_vars = [
    ('num_nodes',  'Number of Nodes'),
    ('num_edges',  'Number of Edges'),
    ('max_degree', 'Maximum Degree'),
    ('avg_degree', 'Average Degree'),
]

In [ ]:
###############################################################################
# TIME-TO-SOLUTION VS GRAPH PROPERTIES (KAPLAN-MEIER MEDIAN)
###############################################################################

for col, label in group_vars:
    if col not in df.columns:
        continue

    # ── Compute KM median TTS per group for each encoding ─────────────────────
    km_oh  = km_tts_by_group(df, col, 'oh_tts_us')
    km_bin = km_tts_by_group(df, col, 'bin_tts_us')

    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, ax = make_square_ax()
    plot_km_tts_pair(ax, km_oh, km_bin, xlabel=label)
    try:
        ax.set_xticks(sorted(df[col].unique()))
    except Exception:
        pass
    fig.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

In [ ]:
###############################################################################
# ONE-HOT VS LOGARITHMIC (PRE-QUADRATIZATION) LOGICAL QUBITS
###############################################################################

fig, ax = make_square_ax()
ax.scatter(df['oh_num_logical'], df['bin_pre_quad'],
           alpha=0.5, edgecolors='k', linewidths=0.3,
           marker='s', s=40, c='#2ca02c')
hi = max(df['oh_num_logical'].max(), df['bin_pre_quad'].max()) * 1.05
ax.plot([0, hi], [0, hi], 'k--', alpha=0.4)
ax.set_xlabel('One-Hot Logical Qubits')
ax.set_ylabel('Logarithmic (Pre-Quadratization) Logical Qubits')
fig.tight_layout()
pdf.savefig(fig)
plt.close(fig)

In [ ]:
###############################################################################
# ONE-HOT VS LOGARITHMIC (POST-QUADRATIZATION) LOGICAL QUBITS
###############################################################################

fig, ax = make_square_ax()
ax.scatter(df['oh_num_logical'], df['bin_post_quad'],
           alpha=0.5, edgecolors='k', linewidths=0.3,
           marker='s', s=40, c='#9467bd')
hi = max(df['oh_num_logical'].max(), df['bin_post_quad'].max()) * 1.05
ax.plot([0, hi], [0, hi], 'k--', alpha=0.4)
ax.set_xlabel('One-Hot Logical Qubits')
ax.set_ylabel('Logarithmic (Post-Quadratization) Logical Qubits')
fig.tight_layout()
pdf.savefig(fig)
plt.close(fig)

In [ ]:
###############################################################################
# ONE-HOT VS LOGARITHMIC PHYSICAL QUBITS (POST-EMBEDDING)
###############################################################################

fig, ax = make_square_ax()
ax.scatter(df_embedding['onehot_total'], df_embedding['logarithmic_total'],
           alpha=0.5, edgecolors='k', linewidths=0.3,
           marker='s', s=40, c='#9467bd')
hi = max(df_embedding['onehot_total'].max(),
         df_embedding['logarithmic_total'].max()) * 1.05
ax.set_xlim(0, hi)
ax.set_ylim(0, hi)
ax.plot([0, hi], [0, hi], 'k--', alpha=0.4)
ax.set_xlabel('One-Hot Physical Qubits')
ax.set_ylabel('Logarithmic (Post-Quadratization) Physical Qubits')
fig.tight_layout()
pdf.savefig(fig)
plt.close(fig)

In [ ]:
###############################################################################
# PHYSICAL QUBITS VS GRAPH PROPERTIES
###############################################################################

for col, label in group_vars:
    if col not in df_embedding.columns:
        continue

    grouped = df_embedding.groupby(col).agg(['mean', 'sem']).reset_index()

    fig, ax = make_square_ax()
    x_oh  = grouped[col]
    y_oh  = grouped[('onehot_total',     'mean')]
    se_oh = grouped[('onehot_total',     'sem')]
    x_bin = grouped[col]
    y_bin = grouped[('logarithmic_total','mean')]
    se_bin= grouped[('logarithmic_total','sem')]
    plot_qubit_pair(ax, x_oh, y_oh, se_oh,
                    x_bin, y_bin, se_bin,
                    xlabel=label, ylabel='Number of Physical Qubits')
    fig.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

In [ ]:
###############################################################################
# AVERAGE CHAIN LENGTH VS NUMBER OF NODES
###############################################################################

cols_to_agg = df_embedding.columns[df_embedding.columns.get_loc('max_deg') + 1:]
grouped = df_embedding.groupby('nodes')[cols_to_agg].agg(['mean', 'sem']).reset_index()

fig, ax = make_square_ax()
x_oh  = grouped['nodes']
y_oh  = grouped[('onehot_average',      'mean')]
se_oh = grouped[('onehot_average',      'sem')]
x_bin = grouped['nodes']
y_bin = grouped[('logarithmic_average', 'mean')]
se_bin= grouped[('logarithmic_average', 'sem')]
plot_qubit_pair(ax, x_oh, y_oh, se_oh,
                x_bin, y_bin, se_bin,
                xlabel='Number of Nodes', ylabel='Average Chain Length')
fig.tight_layout()
pdf.savefig(fig)
plt.close(fig)

In [ ]:
###############################################################################
# CHAIN LENGTH VARIANCE VS NUMBER OF NODES
###############################################################################

cols_to_agg = df_embedding.columns[df_embedding.columns.get_loc('max_deg') + 1:]
grouped = df_embedding.groupby('nodes')[cols_to_agg].agg(['mean', 'sem']).reset_index()

fig, ax = make_square_ax()
x_oh  = grouped['nodes']
y_oh  = grouped[('onehot_variance',      'mean')]
se_oh = grouped[('onehot_variance',      'sem')]
x_bin = grouped['nodes']
y_bin = grouped[('logarithmic_variance', 'mean')]
se_bin= grouped[('logarithmic_variance', 'sem')]
plot_qubit_pair(ax, x_oh, y_oh, se_oh,
                x_bin, y_bin, se_bin,
                xlabel='Number of Nodes', ylabel='Chain Length Variance')
fig.tight_layout()
pdf.savefig(fig)
plt.close(fig)

In [ ]:
###############################################################################
# PHYSICAL QUBITS VS NUMBER OF NODES
###############################################################################

cols_to_agg = df_embedding.columns[df_embedding.columns.get_loc('max_deg') + 1:]
grouped = df_embedding.groupby('nodes')[cols_to_agg].agg(['mean', 'sem']).reset_index()

fig, ax = make_square_ax()
x_oh  = grouped['nodes']
y_oh  = grouped[('onehot_total',      'mean')]
se_oh = grouped[('onehot_total',      'sem')]
x_bin = grouped['nodes']
y_bin = grouped[('logarithmic_total', 'mean')]
se_bin= grouped[('logarithmic_total', 'sem')]
plot_qubit_pair(ax, x_oh, y_oh, se_oh,
                x_bin, y_bin, se_bin,
                xlabel='Number of Nodes', ylabel='Number of Physical Qubits')
fig.tight_layout()
pdf.savefig(fig)
plt.close(fig)

In [ ]:
###############################################################################
# FINALIZE MULTI-PAGE PDF
###############################################################################

pdf.close()
print(f'Plots written to {PDF_OUT}')